# Time Series Forecasting — Neural Network Models

**Week 2 | Perceptron & Multi-Layer Perceptron (MLP)**

| Model | Inputs | Hidden layers | Notes |
|---|---|---|---|
| **Single-layer Perceptron (SLP)** | lagged features | 0 | Linear baseline |
| **Multi-Layer Perceptron (MLP)** | lagged + calendar features | 1–2 | Universal approximator |


---
#### Why do we need neural networks for forecasting?

Statistical models like ARIMA and Holt-Winters make **explicit assumptions** about the data (linearity, stationarity, additive/multiplicative seasonality). When those assumptions hold they are hard to beat. But real-world time series often contain:

- Non-linear relationships between past and future values  
- Complex interactions between multiple covariates  
- Patterns too irregular for a fixed seasonal period  

Neural networks are **model-free** — they learn the mapping from inputs to output directly from data without requiring such assumptions.


---
### 1. Import Libraries

In [ ]:
import warnings; warnings.filterwarnings('ignore')

# ── Data & maths ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# ── Scikit-learn utilities ─────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Neural network (sklearn MLP) ───────────────────────────────────────────────
from sklearn.neural_network import MLPRegressor

# ── For reproducibility ────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print('All libraries loaded successfully.')

---
### 2. Load & Prepare the Data

**Quartely**  
Date column:  →  use `pd.PeriodIndex(year=..., quarter=..., freq='Q').to_timestamp()`  

Frequency: `'QS'` (Quarterly Start), QE, QS-DEC
Seasonal period: **4**

<br>

**Monthly**  
Date column: --> use `pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')`

Frequency: `'MS'`  
Seasonal period: **12**

<br>

**Daily**
Date column: `pd.to_datetime(df['date'])`
Frequency: `'D'`

Seasonal period: **7** (weekly cycle) or **365** (annual cycle)

<br>

**Hourly**
Date column: `pd.to_datetime(df['datetime'])` or `pd.to_datetime(df['date'] + ' ' + df['hour'].astype(str) + ':00')`
Frequency: `'h'`

Seasonal period: **24** (daily cycle), **168** (weekly = 24×7), or **8760** (annual = 24×365)

<br>


**Already, a 'date' variable**  --> use `df['Date'] = pd.to_datetime(df['date'])`

<br>


#### Convert the Data Frame into a  Time Series

```python
ts = df.set_index('Date')['col'].asfreq('MS')
ts = ts.interpolate(limit_direction='both') # To fill in missing values
```
asfreq - https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.asfreq.html


---
### 3. Exploratory Data Analysis & Decomposition

Before modelling, always **look at the data**. Key questions:

- Is there a trend?
- Is the seasonality additive (constant amplitude) or multiplicative (growing amplitude)?
- Are there outliers or structural breaks?

These observations guide:
1. Whether to **log-transform** (stabilises multiplicative variance)
2. How many **lag features** to create
3. Whether to add **calendar features** (month-of-year, quarter)


```python
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(ts.index, ts.values, color='steelblue', linewidth=1.5)
ax.set(title='-------', ylabel='-----', xlabel='Date')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Decompose using multiplicative model (amplitude of seasonality grows with level)
decomp = seasonal_decompose(ts, model='multiplicative', period=seasonal_period)
fig = decomp.plot()
```

#### 4 Log-Transform (Optional but Recommended)

Because the seasonal swings **grow proportionally** with the level (multiplicative seasonality), we apply a **log transform**:

$$
y^* = \ln(y)
$$

This converts multiplicative structure into additive structure and **stabilises variance** — making it easier for the neural network to learn.  
Remember to **inverse-transform** forecasts at the end: $\hat{y} = e^{\hat{y}^*}$.

```python
ts_log = np.log(ts)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].plot(ts.index, ts.values, color='steelblue'); axes[0].set_title('Original')
axes[1].plot(ts_log.index, ts_log.values, color='darkorange'); axes[1].set_title('Log-Transformed')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
```

---
### 5. Feature Engineering — Supervised Windowing

Neural networks do not natively understand sequential order.  
We must manually create the **input features** (lags) and **target** column.

**Lag features** — previous values as predictors:

| Feature | Description |
|---|---|
| `lag_1` | $y_{t-1}$ — most recent observation |
| `lag_2` | $y_{t-2}$ |
| … | … |
| `lag_12` | $y_{t-12}$ — same month last year (captures annual seasonality) |

**Calendar features** (used only in the MLP, not the Perceptron):

| Feature | Description |
|---|---|
| `month_sin`, `month_cos` | Cyclical encoding of month-of-year |

> **Cyclical encoding** avoids the discontinuity between December (12) and January (1)

```python
N_LAGS = 12   # lookback window — try 6 or 24 and see what changes

def make_supervised(series, n_lags, add_calendar=False):
    df = pd.DataFrame({'target': series})

    # add lag features
    for i in range(1, n_lags + 1):
        df[f'lag_{i}'] = series.shift(i)

    # add calendar features (optional)
    if add_calendar:
        m = series.index.month
        df['month_sin'] = np.sin(2 * np.pi * m / 12)
        df['month_cos'] = np.cos(2 * np.pi * m / 12)

    return df.dropna()


# ── Perceptron feature set (lags only) ────────────────────────────────────────
data_slp = make_supervised(ts_log, N_LAGS, add_calendar=False)

# ── MLP feature set (lags + calendar) ─────────────────────────────────────────
data_mlp = make_supervised(ts_log, N_LAGS, add_calendar=True)
```

---
### 6. Train / Test Split & Scaling

#### Temporal Split

For time series, **always split chronologically** — never shuffle!  
Shuffling would leak future information into training and give unrealistically optimistic scores.

```
│◄──────── train (80 %) ────────►│◄── test (20 %) ──►│
```

#### Feature Scaling

Neural networks are sensitive to the **scale** of inputs.  
We use `MinMaxScaler` to map all features to $[0, 1]$:

$$
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

> **Important:** fit the scaler on **training data only**, then apply the same transformation to the test data.  
> Fitting on the full dataset would be **data leakage**.

```python
TEST_SIZE = 24

def split_and_scale(df, test_size):
    # 1. Split data (keep time order)
    train = df.iloc[:-test_size]
    test  = df.iloc[-test_size:]
    test_idx = test.index

    # 2. Separate inputs (X) and target (y)
    X_train = train.drop(columns='target')
    X_test  = test.drop(columns='target')
    y_train = train['target']
    y_test  = test['target']

    # 3. Scale inputs (fit ONLY on train)
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, test_idx


X_train_slp, X_test_slp, y_train_slp, y_test_slp, test_idx = split_and_scale(data_slp, TEST_SIZE)
X_train_mlp, X_test_mlp, y_train_mlp, y_test_mlp, test_idx = split_and_scale(data_mlp, TEST_SIZE)
```


---
### 7. Fit the Models

For each model we document:
- Architecture choices and why
- Expected strengths / weaknesses on this data

We use **`sklearn.neural_network.MLPRegressor`** which implements full backpropagation with configurable layers, activations, and optimisers.  

Full documentation: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html

---

#### The Perceptron — Building Block of All Neural Networks

A single **perceptron** (neuron) computes:

$$
\hat{y} = f\!\left(\sum_{i=1}^{n} w_i x_i + b\right)
$$

| Symbol | Meaning |
|---|---|
| $x_i$ | Input features (e.g. lagged values $y_{t-1}, y_{t-2}, \ldots$) |
| $w_i$ | Learned weights |
| $b$ | Bias term |
| $f(\cdot)$ | Activation function (linear, ReLU, sigmoid, …) |
| $\hat{y}$ | Predicted output |

With a **linear** activation ($f(z) = z$) and lag features as inputs, a single-layer perceptron is mathematically equivalent to an **AR model** — it's a useful sanity-check baseline.


**Common activation functions:**

| Name | Formula | When to use |
|---|---|---|
| Linear | $$f(z) = z$$ | Output layer for regression |
| ReLU | $$f(z) = \max(0, z)$$ | Default hidden-layer choice |
| Tanh | $$f(z) = \tanh(z)$$ | When inputs are centered/normalized |
| Sigmoid | $$f(z) = \frac{1}{1+e^{-z}}$$ | Binary outputs |

#### 8a. Single-Layer Perceptron (SLP)

$$
\hat{y}_t = \mathbf{w} \cdot \mathbf{x}_t + b
$$

**Architecture:** 0 hidden layers, linear activation → pure weighted sum of lag features.  
This is the **neural network equivalent of an AR(12) model**.

| Hyperparameter | Value | Reason |
|---|---|---|
| `hidden_layer_sizes` | `()` | No hidden layers = single perceptron |
| `activation` | `'identity'` | Linear — no non-linearity |
| `solver` | `'lbfgs'` | Fast quasi-Newton solver, good for small datasets |
| `max_iter` | `2000` | Enough epochs for a linear model to converge |

**Expected performance:** Moderate. Should capture the trend encoded in the lags but will struggle with the non-linear seasonal pattern.

```python
slp = MLPRegressor(
    hidden_layer_sizes=(),      # ← empty tuple = no hidden layers
    activation='identity',      # ← linear activation
    solver='lbfgs',
    max_iter=2000,
    random_state=SEED
)
slp.fit(X_train_slp, y_train_slp)
slp_pred_log = slp.predict(X_test_slp)
slp_pred     = np.exp(slp_pred_log)          # inverse log transform

print('SLP learned weights (lag_1 … lag_12):')
print(np.round(slp.coefs_[0].flatten(), 4))
```

#### 8. Multi-Layer Perceptron (MLP) — Shallow

$$
\mathbf{h} = \text{ReLU}(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1), \qquad
\hat{y} = \mathbf{w}_2 \cdot \mathbf{h} + b_2
$$

**Architecture:** 1 hidden layer with 32 neurons, ReLU activation.  

| Hyperparameter | Value | Reason |
|---|---|---|
| `hidden_layer_sizes` | `(32,)` | One hidden layer — enough to capture moderate non-linearity |
| `activation` | `'relu'` | Avoids vanishing gradient; default best-practice |
| `solver` | `'adam'` | Adaptive learning rate; standard for MLPs |
| `learning_rate_init` | `0.001` | Adam default; conservative starting point |
| `max_iter` | `1000` | Monitor loss curve — increase if not converged |
| `early_stopping` | `True` | Stops if validation loss stops improving — prevents overfitting |
| `validation_fraction` | `0.1` | 10 % of training set used for early-stop validation |

**Expected performance:** Better than SLP — the hidden layer lets it model the interaction between trend and seasonality. Inputs include cyclical month features.

```python
mlp_shallow = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=SEED
)

mlp_shallow.fit(X_train_mlp, y_train_mlp)

mshallow_pred_log = mlp_shallow.predict(X_test_mlp)
mshallow_pred     = np.exp(mshallow_pred_log)
mshallow_series   = pd.Series(mshallow_pred, index=test_idx, name='MLP-Shallow')

print(f'Stopped at epoch: {mlp_shallow.n_iter_}')
print(f'Best validation loss: {mlp_shallow.best_validation_score_:.6f}')
```

#### 9. Multi-Layer Perceptron (MLP) — Deep

---

#### Multi-Layer Perceptron (MLP)

Stack multiple layers of perceptrons and add **non-linear** activations between them:

$$
\begin{aligned}
\mathbf{h}^{(1)} &= f_1(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}) \\
\mathbf{h}^{(2)} &= f_2(\mathbf{W}^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)}) \\
\hat{y} &= \mathbf{w}^{(\text{out})} \cdot \mathbf{h}^{(2)} + b^{(\text{out})}
\end{aligned}
$$

The universal approximation theorem guarantees that an MLP with at least one hidden layer and enough neurons can approximate **any continuous function** — including complex temporal patterns.


#### How does the network learn?

Training minimises a **loss function** (MSE for regression) using **backpropagation** + **gradient descent**:

$$
\mathcal{L} = \frac{1}{N}\sum_{t}(y_t - \hat{y}_t)^2
\qquad
w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w}
$$

| Hyperparameter | Effect |
|---|---|
| **Learning rate** $\eta$ | Step size; too large → diverge, too small → slow |
| **Epochs** | Passes through the full training set |
| **Hidden units** | Model capacity; more → risk of overfitting |
| **Batch size** | Samples per gradient update |


$$
\mathbf{h}^{(1)} = \text{ReLU}(\mathbf{W}_1\mathbf{x}+\mathbf{b}_1),\;
\mathbf{h}^{(2)} = \text{ReLU}(\mathbf{W}_2\mathbf{h}^{(1)}+\mathbf{b}_2),\;
\hat{y} = \mathbf{w}_3 \cdot \mathbf{h}^{(2)} + b_3
$$

**Architecture:** 2 hidden layers (64 → 32 neurons), ReLU, Adam with L2 regularisation.

| Hyperparameter | Value | Reason |
|---|---|---|
| `hidden_layer_sizes` | `(64, 32)` | Funnel shape: wider early layer for feature extraction |
| `alpha` | `0.001` | L2 regularisation weight — prevents overfitting on 120 training samples |
| `early_stopping` | `True` | Extra guard against overfitting |

**Expected performance:** More capacity to model complex patterns. Risk: **overfitting** on only ~100 training samples. Regularisation + early stopping mitigate this.

```python
mlp_deep = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,                 # L2 regularisation
    learning_rate_init=0.001,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=SEED
)

mlp_deep.fit(X_train_mlp, y_train_mlp)

mdeep_pred_log = mlp_deep.predict(X_test_mlp)
mdeep_pred     = np.exp(mdeep_pred_log)
mdeep_series   = pd.Series(mdeep_pred, index=test_idx, name='MLP-Deep')

print(f'Stopped at epoch: {mlp_deep.n_iter_}')
print(f'Best validation loss: {mlp_deep.best_validation_score_:.6f}')
```

---
### 10. Visualise Results

Plot the full series (train + actual test) against all three model forecasts.

```python
# Reconstruct the original (non-log) training portion for plotting
ts_original = np.exp(ts_log)   # same as ts, just consistent variable name
train_plot = ts_original.iloc[:len(ts_original) - TEST_SIZE]
test_plot  = ts_original.iloc[-TEST_SIZE:]

fig, ax = plt.subplots(figsize=(13, 5))

#ax.plot(train_plot.index, train_plot.values, label='Train',  color='steelblue', linewidth=1.5)
ax.plot(test_plot.index,  test_plot.values,  label='Actual', color='black', linewidth=2, linestyle='--')

ax.plot(slp_series.index,      slp_series.values,      label='SLP',         color='tomato',      linewidth=1.5)
ax.plot(mshallow_series.index, mshallow_series.values, label='MLP-Shallow', color='darkorange',  linewidth=1.5)
ax.plot(mdeep_series.index,    mdeep_series.values,    label='MLP-Deep',    color='mediumorchid', linewidth=1.5)
#ax.plot(mdeep_series.index,    auto_arima_forecast.values,    label='Auto-ARIMA',    color='blue', linewidth=1.5)
#ax.plot(mdeep_series.index,    hw_forecast.values,    label='Holt',    color='black', linewidth=1.5)

ax.axvline(test_plot.index[0], color='grey', linestyle=':', linewidth=1, label='Train/Test split')
ax.set(title='------',
       ylabel='------', xlabel='Date')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```

---
### 11. Loss Curve — Training Diagnostics

Plotting the **training loss curve** tells you whether the model has converged or is still learning.

| Pattern | Diagnosis |
|---|---|
| Still decreasing at end | Increase `max_iter` |
| Flat plateau early | Try a larger learning rate or larger network |
| Train loss low, validation loss high | **Overfitting** — add regularisation or reduce model size |
| Wild oscillations | Learning rate too high — reduce `learning_rate_init` |

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, name, color in [
    (axes[0], mlp_shallow, 'MLP-Shallow', 'darkorange'),
    (axes[1], mlp_deep,    'MLP-Deep',    'mediumorchid')
]:
    ax.plot(model.loss_curve_, label='Train loss', color=color)
    if hasattr(model, 'validation_scores_'):
        # sklearn stores val scores (higher = better) so negate for display
        ax.plot([-v for v in model.validation_scores_],
                label='Val loss (negated)', color='steelblue', linestyle='--')
    ax.set(title=f'{name} — Loss Curve', xlabel='Epoch', ylabel='MSE Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
```

---
### 12. Comparison with Week 1 Statistical Models

If not installed - pip install pmdarima

```python
# Train/test split
test_size = 30
train = ts.iloc[:-test_size]
test = ts.iloc[-test_size:]
size = len(test)

# Auto ARIMA
auto_arima_model = pm.auto_arima(
    train,
    seasonal=True,
    m=5,                  # weekly pattern in business days
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)

print(auto_arima_model.summary())

auto_arima_forecast = pd.Series(
    auto_arima_model.predict(n_periods=size),
    index=test.index,
    name='Auto-ARIMA'
)

# Holt-Winters
hw_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=5
).fit(optimized=True)

hw_forecast = pd.Series(
    hw_model.forecast(size),
    index=test.index,
    name='Holt-Winters'
)
```

---
### 13. Error Metrics

We use the same evaluation function format as Week 1 so results are directly comparable.

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_t - \hat{y}_t)^2}$ | In original units; penalises large errors more |
| **MAE** | $\frac{1}{n}\sum|y_t - \hat{y}_t|$ | In original units; robust to outliers |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_t - \hat{y}_t}{y_t}\right|$ | Percentage error; scale-independent |

```python
def evaluate(name: str, actual: np.ndarray, predicted: np.ndarray) -> dict:
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {'Model': name,
            'RMSE':  round(rmse, 2),
            'MAE':   round(mae,  2),
            'MAPE':  round(mape, 2)}


actual = test_plot.values   # original (non-log) scale

results = pd.DataFrame([
    evaluate('SLP',           actual, slp_series.values),
    evaluate('MLP-Shallow',   actual, mshallow_series.values),
    evaluate('MLP-Deep',      actual, mdeep_series.values),
    evaluate('Auto-ARIMA',    actual, auto_arima_forecast.values),
    evaluate('Holt-Winters',  actual, hw_forecast.values),
]).sort_values('RMSE').reset_index(drop=True)

print('\n=== Forecast Error Summary ===')
print(results.to_string(index=False))
```

---
### 14. Hyperparameter Sensitivity

One of the key challenges with neural networks is that performance depends heavily on **hyperparameter choices**.  
The cell below runs a small grid search over hidden layer sizes and lag window sizes.

```python
from itertools import product
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

lag_options = [6, 12, 24]
layer_options = [(16,), (32,), (64, 32)]

results = []

for n_lags, layers in product(lag_options, layer_options):
    # build supervised data for this lag choice
    df_supervised = make_supervised(ts_log, n_lags, add_calendar=True)

    # split and scale
    X_train, X_test, y_train, y_test,test_idx  = split_and_scale(df_supervised, TEST_SIZE)

    # fit model
    model = MLPRegressor(hidden_layer_sizes=layers, activation='relu', solver='adam', alpha=0.001, max_iter=1000, early_stopping=True, validation_fraction=0.1, random_state=SEED)
    model.fit(X_train, y_train)

    # predict and convert back from log scale
    preds = np.exp(model.predict(X_test))
    actual = np.exp(y_test)

    rmse = np.sqrt(mean_squared_error(actual, preds))

    results.append({'n_lags': n_lags,'layers': layers,'RMSE': round(rmse, 2)})

grid_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)

print('=== Grid Search Results ===')
print(grid_df.to_string(index=False))
```